In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_24 = pd.read_csv('/content/drive/MyDrive/2024_시간대_전체_20260618_36743.csv', encoding='CP949')
df_19 = pd.read_csv('/content/drive/MyDrive/2019_시간대_전체_20260618_36743.csv', encoding='CP949')

/tmp/ipykernel_462/4105125755.py:1: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,108) have mixed types. Specify dtype option on import or set low_memory=False.
  df_24 = pd.read_csv('/content/drive/MyDrive/2024_시간대_전체_20260618_36743.csv', encoding='CP949')
/tmp/ipykernel_462/4105125755.py:2: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_19 = pd.read_csv('/content/drive/MyDrive/2019_시간대_전체_20260618_36743.csv', encoding='CP949')


In [ ]:
# 데이터 병합
df = pd.concat([df_24, df_19], ignore_index=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102376 entries, 0 to 102375
Columns: 112 entries, (주행동시간대) 오전 06:00 to 평토일구분코드
dtypes: int64(3), object(109)
memory usage: 87.5+ MB


In [ ]:
df.head()

,(주행동시간대) 오전 06:00,(주행동시간대) 오전 06:10,(주행동시간대) 오전 06:20,(주행동시간대) 오전 06:30,(주행동시간대) 오전 06:40,(주행동시간대) 오전 06:50,(주행동시간대) 오전 07:00,(주행동시간대) 오전 07:10,(주행동시간대) 오전 07:20,(주행동시간대) 오전 07:30,...,(주행동시간대) 오후 11:00,(주행동시간대) 오후 11:10,(주행동시간대) 오후 11:20,(주행동시간대) 오후 11:30,(주행동시간대) 오후 11:40,(주행동시간대) 오후 11:50,(주행동시간대) 오전 00:00,요일구분코드,회차,평토일구분코드
0,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,...,7112015,7112015,7112015,1222011,1222011,1222011,7112015,4,6,1
1,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,...,1222011,8421011,8421011,8421011,8421011,8421011,8431011,5,6,1
2,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,...,8421011,8421011,8421011,8421011,8421011,8421011,8421011,6,6,2
3,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,...,72*2011,72*2011,72*2011,72*2011,72*2011,72*2011,8992011,5,8,1
4,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,1112011,...,1412011,1412011,1412011,1112011,1112011,1112011,8421011,0,5,3


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 타겟 행동 분류 세팅
# ---------------------------------------------------------
target_actions = {
    '식사하기': ['121****', '122****'],
    '의료서비스 받기': ['133****'],
    '개인위생 및 외모 관리': ['141****', '142****', '143****', '149****'],
    '출근 및 업무 활동': ['210****', '221****', '222****', '223****', '229****', '241****', '242****', '249****', '252****'],
    '학교 활동': ['311****', '312****', '313****', '314****', '319****'],
    '학원 수강': ['321****'],
    '공부 활동': ['322****', '323****', '329****'],
    '자녀와 놀아주기, 스포츠활동 하기': ['515****', '525****'],
    '자녀교육 관련 참여': ['516****', '526****', '722****'],
    '자원봉사': ['611****', '612****', '613****', '614****', '619****', '621****', '622****', '629****'],
    '문화 행사 참여': ['723****'],
    '종교 활동': ['731****', '732****', '739****'],
    '영화 관람': ['811****'],
    '연극‧콘서트 등 공연 관람': ['812****'],
    '전시관‧박물관 관람': ['813****'],
    '스포츠 경기 관람': ['814****'],
    '관광‧드라이브': ['815****', '819****'],
    '걷기‧산책': ['831****'],
    '운동': ['832****', '833****', '834****', '835****', '836****', '839****'],
    '낚시‧사냥': ['837****']
}

# 가사일 데이터 세팅
houseworks = {
    '설거지‧식후 정리': ['413****'],
    '세탁하기': ['421****'],
    '세탁물 건조': ['422****'],
    '청소': ['431****']
}

# ---------------------------------------------------------
# 2. 분석에 사용할 데이터만 Numpy 배열로 추출
# ---------------------------------------------------------
start_col = '(주행동시간대) 오전 06:00'
end_col = '(주행동시간대) 오전 00:00'

time_columns = df.loc[:, start_col:end_col]
arr = time_columns.values.astype(str)

num_cols = arr.shape[1]
col_indices = np.arange(num_cols)

def create_mask(data_array, codes):
    """
    여러 개의 코드와 와일드카드(*)를 해석하여 True/False 마스크를 반환하는 함수
    """
    mask = np.zeros(data_array.shape, dtype=bool)

    for code in codes:
        if '*' in code:
            prefix = code.replace('*', '')
            mask |= np.char.startswith(data_array, prefix)
        else:
            mask |= (data_array == code)

    return mask

# ---------------------------------------------------------
# 3. 고속 행렬 연산을 통한 집계 (tqdm 및 조건부 가사일 필터링 적용)
# ---------------------------------------------------------
results = []

# 특별히 필터링할 타겟 행동과 가사일 리스트 정의
sports_actions = ['운동', '걷기‧산책', '낚시‧사냥']
laundry_works = ['세탁하기', '세탁물 건조']

for t_name, t_codes in tqdm(target_actions.items(), desc="데이터 분석 진행 중"):

    t_mask = create_mask(arr, t_codes)
    t_exists = t_mask.any(axis=1)

    if not t_exists.any():
        continue

    t_first_idx = np.argmax(t_mask, axis=1)

    for h_name, h_codes in houseworks.items():

        # 🔥 조건 1: '식사하기'가 아닌 경우 '설거지‧식후 정리'는 제외
        if t_name != '식사하기' and h_name == '설거지‧식후 정리':
            continue

        # 🔥 조건 2: '운동, 걷기‧산책, 낚시‧사냥'인 경우 '세탁하기, 세탁물 건조'가 아니면 제외
        if t_name in sports_actions and h_name not in laundry_works:
            continue

        h_mask = create_mask(arr, h_codes)
        after_mask = col_indices > t_first_idx[:, None]
        valid_rows = t_exists & (h_mask & after_mask).any(axis=1)
        count = valid_rows.sum()

        if count > 0:
            results.append({
                '조건_행동': t_name,
                '이후_수행된_가사일': h_name,
                '발생_횟수(일수)': count
            })

# ---------------------------------------------------------
# 4. 결과 출력
# ---------------------------------------------------------
result_df = pd.DataFrame(results)

if not result_df.empty:
    top_trends = result_df.sort_values('발생_횟수(일수)', ascending=False)\
                          .groupby('조건_행동').head(1)\
                          .reset_index(drop=True)

    print("\n=== 특정 행동 이후 가장 많이 수행된 가사일 1위 ===")
    print(top_trends)
else:
    print("\n조건에 맞는 데이터가 없습니다.")

데이터 분석 진행 중: 100%|██████████| 20/20 [00:14<00:00,  1.33it/s]


=== 특정 행동 이후 가장 많이 수행된 가사일 1위 ===
                 조건_행동 이후_수행된_가사일  발생_횟수(일수)
0                 식사하기  설거지‧식후 정리      46249
1         개인위생 및 외모 관리         청소      29322
2           출근 및 업무 활동         청소       6450
3                   운동       세탁하기       2756
4                종교 활동         청소       2215
5                걷기‧산책       세탁하기       1889
6             의료서비스 받기         청소       1212
7   자녀와 놀아주기, 스포츠활동 하기         청소        959
8                공부 활동         청소        717
9                학교 활동         청소        170
10               학원 수강         청소        138
11          자녀교육 관련 참여         청소         57
12               낚시‧사냥       세탁하기         10


In [ ]:
result_df

,조건_행동,이후_수행된_가사일,발생_횟수(일수)
0,식사하기,설거지‧식후 정리,46249
1,식사하기,세탁하기,16813
2,식사하기,세탁물 건조,11006
3,식사하기,청소,30314
4,의료서비스 받기,세탁하기,605
5,의료서비스 받기,세탁물 건조,459
6,의료서비스 받기,청소,1212
7,개인위생 및 외모 관리,세탁하기,16412
8,개인위생 및 외모 관리,세탁물 건조,10426
9,개인위생 및 외모 관리,청소,29322


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 환경 및 데이터 세팅
# ---------------------------------------------------------
target_actions = {
    '수면': ['111****', '112****'],
    '식사하기': ['121****', '122****'],
    '의료서비스 받기': ['133****'],
    '개인위생 및 외모 관리': ['141****', '142****', '143****', '149****'],
    '출근 및 업무 활동': ['210****', '221****', '222****', '223****', '229****', '241****', '242****', '249****', '252****'],
    '학교 활동': ['311****', '312****', '313****', '314****', '319****'],
    '학원 수강': ['321****'],
    '공부 활동': ['322****', '323****', '329****'],
    '자녀와 놀아주기, 스포츠활동 하기': ['515****', '525****'],
    '자녀교육 관련 참여': ['516****', '526****', '722****'],
    '자원봉사': ['611****', '612****', '613****', '614****', '619****', '621****', '622****', '629****'],
    '문화 행사 참여': ['723****'],
    '종교 활동': ['731****', '732****', '739****'],
    '영화 관람': ['811****'],
    '연극‧콘서트 등 공연 관람': ['812****'],
    '전시관‧박물관 관람': ['813****'],
    '스포츠 경기 관람': ['814****'],
    '관광‧드라이브': ['815****', '819****'],
    '걷기‧산책': ['831****'],
    '운동': ['832****', '833****', '834****', '835****', '836****', '839****'],
    '낚시‧사냥': ['837****']
}

houseworks = {
    '설거지‧식후 정리': ['413****'],
    '세탁하기': ['421****'],
    '세탁물 건조': ['422****'],
    '청소': ['431****']
}

# 데이터 추출
start_col = '(주행동시간대) 오전 06:00'
end_col = '(주행동시간대) 오전 00:00'
time_columns = df.loc[:, start_col:end_col]
arr = time_columns.values.astype(str)

num_cols = arr.shape[1]
col_indices = np.arange(num_cols)
total_days = arr.shape[0]

def create_mask(data_array, codes):
    mask = np.zeros(data_array.shape, dtype=bool)
    for code in codes:
        if '*' in code:
            prefix = code.replace('*', '')
            mask |= np.char.startswith(data_array, prefix)
        else:
            mask |= (data_array == code)
    return mask

# ---------------------------------------------------------
# 2. 가사일(B)의 기저 확률(Base Rate) 사전 계산
# ---------------------------------------------------------
base_rates = {}
for h_name, h_codes in houseworks.items():
    h_mask_overall = create_mask(arr, h_codes)
    h_exists_overall = h_mask_overall.any(axis=1)
    base_rates[h_name] = h_exists_overall.sum() / total_days

# ---------------------------------------------------------
# 3. 고속 행렬 연산 및 Lift 지표 도출 (Time Window 추가)
# ---------------------------------------------------------
results = []
sports_actions = ['운동', '걷기‧산책', '낚시‧사냥']
laundry_works = ['세탁하기', '세탁물 건조']

# 🔥 새롭게 추가된 시간 창 변수 (단위: 10분)
# 12칸 = 120분 = 2시간 이내에 수행된 행동만 유효한 것으로 봅니다.
window_size = 12

for t_name, t_codes in tqdm(target_actions.items(), desc="SOTA 분석 (2시간 Time Window) 진행 중"):

    t_mask = create_mask(arr, t_codes)
    t_exists = t_mask.any(axis=1)
    t_count = t_exists.sum()

    if t_count == 0:
        continue

    t_first_idx = np.argmax(t_mask, axis=1)

    for h_name, h_codes in houseworks.items():

        if t_name != '식사하기' and h_name == '설거지‧식후 정리':
            continue
        if t_name in sports_actions and h_name not in laundry_works:
            continue

        h_mask = create_mask(arr, h_codes)

        # 🔥 핵심 로직 변경: 시작점(이후)부터 끝점(2시간 이내)까지만 True로 만듭니다.
        after_mask = (col_indices > t_first_idx[:, None]) & (col_indices <= t_first_idx[:, None] + window_size)

        # 행동 A 발생 & 그 이후 '2시간 이내'에 B 발생
        valid_rows = t_exists & (h_mask & after_mask).any(axis=1)
        joint_count = valid_rows.sum()

        if joint_count > 0:
            confidence = joint_count / t_count
            base_rate = base_rates[h_name]
            lift = confidence / base_rate if base_rate > 0 else 0

            results.append({
                '조건_행동(A)': t_name,
                '이후_가사일(B)': h_name,
                'A_수행일수': t_count,
                '2시간내_B수행일수': joint_count,
                '신뢰도(Confidence)': round(confidence, 4),
                '평소_B수행확률(Base Rate)': round(base_rate, 4),
                '향상도(Lift)': round(lift, 2)
            })

# ---------------------------------------------------------
# 4. 결과 출력
# ---------------------------------------------------------
result_df = pd.DataFrame(results)

if not result_df.empty:
    top_trends_by_lift = result_df.sort_values('향상도(Lift)', ascending=False)\
                                  .groupby('조건_행동(A)').head(1)\
                                  .reset_index(drop=True)

    print("\n=== [SOTA] 타겟 행동 직후 '2시간 이내' 가사일 유발력(Lift) 1위 ===")
    print(top_trends_by_lift[['조건_행동(A)', '이후_가사일(B)', '신뢰도(Confidence)', '향상도(Lift)']])
else:
    print("\n조건에 맞는 데이터가 없습니다.")

SOTA 분석 (2시간 Time Window) 진행 중: 100%|██████████| 21/21 [00:16<00:00,  1.29it/s]


=== [SOTA] 타겟 행동 직후 '2시간 이내' 가사일 유발력(Lift) 1위 ===
              조건_행동(A)  이후_가사일(B)  신뢰도(Confidence)  향상도(Lift)
0                 식사하기  설거지‧식후 정리           0.2616       0.57
1           자녀교육 관련 참여       세탁하기           0.0560       0.29
2         개인위생 및 외모 관리         청소           0.0840       0.25
3   자녀와 놀아주기, 스포츠활동 하기         청소           0.0559       0.17
4                종교 활동         청소           0.0418       0.13
5                   운동       세탁하기           0.0238       0.12
6                걷기‧산책       세탁하기           0.0203       0.11
7                   수면         청소           0.0232       0.07
8             의료서비스 받기         청소           0.0227       0.07
9                공부 활동     세탁물 건조           0.0052       0.05
10          출근 및 업무 활동     세탁물 건조           0.0018       0.02
11               학원 수강         청소           0.0009       0.00
12               학교 활동         청소           0.0003       0.00


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 환경 및 데이터 세팅 (기존과 동일)
# ---------------------------------------------------------
target_actions = {
    '수면': ['111****', '112****'], '식사하기': ['121****', '122****'],
    '의료서비스 받기': ['133****'], '개인위생 및 외모 관리': ['141****', '142****', '143****', '149****'],
    '출근 및 업무 활동': ['210****', '221****', '222****', '223****', '229****', '241****', '242****', '249****', '252****'],
    '학교 활동': ['311****', '312****', '313****', '314****', '319****'], '학원 수강': ['321****'],
    '공부 활동': ['322****', '323****', '329****'], '자녀와 놀아주기, 스포츠활동 하기': ['515****', '525****'],
    '자녀교육 관련 참여': ['516****', '526****', '722****'],
    '자원봉사': ['611****', '612****', '613****', '614****', '619****', '621****', '622****', '629****'],
    '문화 행사 참여': ['723****'], '종교 활동': ['731****', '732****', '739****'],
    '영화 관람': ['811****'], '연극‧콘서트 등 공연 관람': ['812****'], '전시관‧박물관 관람': ['813****'],
    '스포츠 경기 관람': ['814****'], '관광‧드라이브': ['815****', '819****'],
    '걷기‧산책': ['831****'], '운동': ['832****', '833****', '834****', '835****', '836****', '839****'],
    '낚시‧사냥': ['837****']
}

houseworks = {
    '설거지‧식후 정리': ['413****'], '세탁하기': ['421****'],
    '세탁물 건조': ['422****'], '청소': ['431****']
}

start_col = '(주행동시간대) 오전 06:00'
end_col = '(주행동시간대) 오전 00:00'
time_columns = df.loc[:, start_col:end_col]
arr = time_columns.values.astype(str)

num_cols = arr.shape[1]
col_indices = np.arange(num_cols)
total_days = arr.shape[0]

def create_mask(data_array, codes):
    mask = np.zeros(data_array.shape, dtype=bool)
    for code in codes:
        if '*' in code:
            prefix = code.replace('*', '')
            mask |= np.char.startswith(data_array, prefix)
        else:
            mask |= (data_array == code)
    return mask

# ---------------------------------------------------------
# 2. 가사일(B)의 기저 확률(Base Rate) 사전 계산 (🔥 핵심 교정 부분)
# ---------------------------------------------------------
window_size = 18 # 2시간 (10분 x 12칸)
base_rates = {}

for h_name, h_codes in houseworks.items():
    h_mask_overall = create_mask(arr, h_codes)

    # 1) 10분(1칸) 단위의 아주 미세한 발생 확률 계산
    total_slots = total_days * num_cols
    b_occurrences = h_mask_overall.sum()
    prob_b_single_slot = b_occurrences / total_slots

    # 2) '임의의 2시간(12칸)' 동안 해당 가사일이 발생할 확률로 보정
    # 공식: 1 - (한 번도 안 할 확률)^12
    adjusted_base_rate = 1 - ((1 - prob_b_single_slot) ** window_size)

    base_rates[h_name] = adjusted_base_rate

# ---------------------------------------------------------
# 3. 고속 행렬 연산 및 Lift 지표 도출
# ---------------------------------------------------------
results = []
sports_actions = ['운동', '걷기‧산책', '낚시‧사냥']
laundry_works = ['세탁하기', '세탁물 건조']

for t_name, t_codes in tqdm(target_actions.items(), desc="SOTA 분석 (동기화된 2hr Time Window)"):

    t_mask = create_mask(arr, t_codes)
    t_exists = t_mask.any(axis=1)
    t_count = t_exists.sum()

    if t_count == 0:
        continue

    t_first_idx = np.argmax(t_mask, axis=1)

    for h_name, h_codes in houseworks.items():

        if t_name != '식사하기' and h_name == '설거지‧식후 정리':
            continue
        if t_name in sports_actions and h_name not in laundry_works:
            continue

        h_mask = create_mask(arr, h_codes)

        after_mask = (col_indices > t_first_idx[:, None]) & (col_indices <= t_first_idx[:, None] + window_size)
        valid_rows = t_exists & (h_mask & after_mask).any(axis=1)
        joint_count = valid_rows.sum()

        if joint_count > 0:
            confidence = joint_count / t_count
            base_rate = base_rates[h_name]
            lift = confidence / base_rate if base_rate > 0 else 0

            results.append({
                '조건_행동(A)': t_name,
                '이후_가사일(B)': h_name,
                '신뢰도(2시간내)': round(confidence, 4),
                '보정된_기저확률(2시간)': round(base_rate, 4),
                '향상도(Lift)': round(lift, 2)
            })

# ---------------------------------------------------------
# 4. 결과 출력
# ---------------------------------------------------------
result_df = pd.DataFrame(results)

if not result_df.empty:
    top_trends_by_lift = result_df.sort_values('향상도(Lift)', ascending=False)\
                                  .groupby('조건_행동(A)').head(1)\
                                  .reset_index(drop=True)

    print("\n=== [SOTA 완결판] 타겟 행동 직후 '2시간 이내' 가사일 유발력(Lift) 1위 ===")
    print(top_trends_by_lift[['조건_행동(A)', '이후_가사일(B)', '신뢰도(2시간내)', '보정된_기저확률(2시간)', '향상도(Lift)']])
else:
    print("\n조건에 맞는 데이터가 없습니다.")

SOTA 분석 (동기화된 2hr Time Window): 100%|██████████| 21/21 [00:16<00:00,  1.25it/s]


=== [SOTA 완결판] 타겟 행동 직후 '2시간 이내' 가사일 유발력(Lift) 1위 ===
              조건_행동(A) 이후_가사일(B)  신뢰도(2시간내)  보정된_기저확률(2시간)  향상도(Lift)
0           자녀교육 관련 참여      세탁하기     0.0862         0.0667       1.29
1                 식사하기      세탁하기     0.0745         0.0667       1.12
2         개인위생 및 외모 관리      세탁하기     0.0638         0.0667       0.96
3   자녀와 놀아주기, 스포츠활동 하기    세탁물 건조     0.0307         0.0367       0.84
4                종교 활동      세탁하기     0.0449         0.0667       0.67
5                   운동      세탁하기     0.0425         0.0667       0.64
6                걷기‧산책      세탁하기     0.0386         0.0667       0.58
7                   수면      세탁하기     0.0298         0.0667       0.45
8             의료서비스 받기    세탁물 건조     0.0162         0.0367       0.44
9                공부 활동    세탁물 건조     0.0067         0.0367       0.18
10          출근 및 업무 활동      세탁하기     0.0058         0.0667       0.09
11               학교 활동        청소     0.0017         0.1946       0.01
12               학원 수강        청소   